# MLP temporal para predicción de yield

Mismo esquema de validación que `ridge_temporal.py` (ventana creciente: entrenar con años < t, probar en t), para poder comparar el MLP contra el ridge línea a línea.

- **X:** genotipos (-1/0/1) de los SNPs con < 20% de NA, imputados con la media del SNP y estandarizados.
- **y:** yield ajustado por ambiente y tester (BLUE, de `build_dataset.py`), centrado por año.
- **Selección de épocas:** para cada año de prueba t se entrena con años < t-1, se mide en t-1 para elegir el número de épocas, y luego se reentrena con años < t. El año t nunca se usa para decidir nada.
- **Prerrequisito:** haber corrido `build_dataset.py` (genera `data/processed/prediction/`) y `ridge_temporal.py` (para la comparación).

Empieza con `QUICK_RUN = True` para verificar que todo corre; después ponlo en `False`.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


def find_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "DIGITALTOOL").is_dir():
            return d
    raise FileNotFoundError("Run the notebook from inside the Hackolombia repo")


BASE_DIR = find_root(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "DIGITALTOOL" / "data" / "processed" / "prediction"
sys.path.insert(0, str(BASE_DIR / "DIGITALTOOL" / "src" / "03_prediction"))
from ridge_temporal import pearson, metrics  # same metrics as the ridge baseline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__, "| data:", DATA_DIR)

In [ ]:
QUICK_RUN = True   # True: only test year 2008, 1 seed, few epochs (smoke test)

CFG = dict(
    max_snp_na=0.20,
    test_years=[2008] if QUICK_RUN else list(range(2002, 2009)),
    hidden=(256, 64),
    dropout=0.3,
    input_dropout=0.1,
    lr=1e-3,
    weight_decay=1e-2,
    batch_size=512,
    max_epochs=5 if QUICK_RUN else 40,
    patience=5,
    n_seeds=1 if QUICK_RUN else 3,
    use_weights=True,      # weight lines by number of observations
    weight_cap=10,
)
CFG

## 1. Datos

In [ ]:
X_raw = np.load(DATA_DIR / "X.npy", mmap_mode="r")
lines = pd.read_parquet(DATA_DIR / "lines.parquet")

keep = np.where(np.isnan(X_raw).mean(axis=0) < CFG["max_snp_na"])[0]
X = np.asarray(X_raw[:, keep], dtype=np.float32)
print(f"{X.shape[0]} lines, {X.shape[1]}/{X_raw.shape[1]} SNPs kept")

# Mean imputation + standardization using pre-2008 lines only (2008 is never used for fitting)
years = lines["YEAR"].to_numpy()
pre = years < 2008
mu = np.nanmean(X[pre], axis=0)
for j in range(X.shape[1]):
    col = X[:, j]
    col[np.isnan(col)] = mu[j]
X -= mu
sd = X[pre].std(axis=0)
sd[sd < 1e-6] = 1.0
X /= sd
print("NaN left:", int(np.isnan(X).sum()))

In [ ]:
# Target: BLUE centered by year, scaled by the pre-2008 sd
lines["y"] = lines["BLUE"] - lines.groupby("YEAR")["BLUE"].transform("mean")
lines["y"] = lines["y"] / lines.loc[pre, "y"].std()

w = np.clip(lines["N_OBS"].to_numpy(dtype=np.float32), 1, CFG["weight_cap"]) if CFG["use_weights"] \
    else np.ones(len(lines), dtype=np.float32)
w = (w / w.mean()).astype(np.float32)

y = lines["y"].to_numpy(dtype=np.float32)
pops = lines["POP"].to_numpy()
X_t = torch.from_numpy(X)
print(lines.groupby("YEAR").size().to_dict())

## 2. Modelo y entrenamiento

In [ ]:
def make_model(p):
    layers = [nn.Dropout(CFG["input_dropout"])]
    d = p
    for h in CFG["hidden"]:
        layers += [nn.Linear(d, h), nn.GELU(), nn.Dropout(CFG["dropout"])]
        d = h
    layers.append(nn.Linear(d, 1))
    return nn.Sequential(*layers)


@torch.no_grad()
def predict(model, Xq):
    model.eval()
    out = [model(Xq[i:i + 4096].to(DEVICE)).squeeze(1).cpu() for i in range(0, len(Xq), 4096)]
    return torch.cat(out).numpy()


def train_model(tr_idx, val_idx=None, epochs=None, seed=0):
    """Train on lines tr_idx. With val_idx, early-stop on validation Pearson r and
    return the best epoch; with epochs, train exactly that many epochs."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    Xtr = X_t[torch.from_numpy(tr_idx)].to(DEVICE)
    ytr = torch.from_numpy(y[tr_idx]).to(DEVICE)
    wtr = torch.from_numpy(w[tr_idx]).to(DEVICE)
    if val_idx is not None:
        Xva, yva = X_t[torch.from_numpy(val_idx)], y[val_idx]

    model = make_model(X.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    bs, n = CFG["batch_size"], len(tr_idx)
    max_ep = epochs if epochs is not None else CFG["max_epochs"]
    best_r, best_ep, hist = -np.inf, 1, []

    for ep in range(1, max_ep + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, bs):
            b = perm[i:i + bs]
            loss = (wtr[b] * (model(Xtr[b]).squeeze(1) - ytr[b]) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        if val_idx is not None:
            r = pearson(predict(model, Xva), yva)
            hist.append(r)
            if r > best_r:
                best_r, best_ep = r, ep
            elif ep - best_ep >= CFG["patience"]:
                break
    return model, best_ep, hist

## 3. Ventana creciente

Para cada año de prueba t: (1) por semilla, entrenar con años < t-1 y validar en t-1 para elegir épocas; (2) reentrenar con años < t con esa cantidad de épocas y promediar las semillas. Puede tardar de decenas de minutos a más de una hora en CPU con `QUICK_RUN = False`.

In [ ]:
rows, preds_2008 = [], None
for t in CFG["test_years"]:
    t0 = time.time()
    tr_prev = np.where(years < t - 1)[0]
    val = np.where(years == t - 1)[0]
    tr_all = np.where(years < t)[0]
    te = np.where(years == t)[0]

    best_eps = []
    for s in range(CFG["n_seeds"]):
        _, ep, _ = train_model(tr_prev, val_idx=val, seed=s)
        best_eps.append(ep)
    n_ep = int(np.median(best_eps))

    seed_preds = []
    for s in range(CFG["n_seeds"]):
        model, _, _ = train_model(tr_all, epochs=n_ep, seed=100 + s)
        seed_preds.append(predict(model, X_t[torch.from_numpy(te)]))
    pred = np.mean(seed_preds, axis=0)

    res = metrics(pred, y[te].astype(np.float64), pops[te])
    res.update(test_year=t, n_epochs=n_ep, n_train=len(tr_all), n_test=len(te))
    rows.append(res)
    if t == 2008:
        preds_2008 = pd.DataFrame({"LINE_ID": lines["LINE_ID"].to_numpy()[te], "POP": pops[te],
                                   "BLUE": lines["BLUE"].to_numpy()[te], "pred": pred})
    print(f"test {t}: epochs={n_ep} (per seed {best_eps}) | r_all={res['r_all']:.3f} "
          f"r_within={res['r_within']:.3f} gain_all={res['gain_all']:.3f} "
          f"gain_within={res['gain_within']:.3f} | {time.time() - t0:.0f}s")

mlp = pd.DataFrame(rows)

## 4. Comparación con el ridge

Ojo: aquí `y` está escalada por su desviación estándar, así que las **ganancias** del MLP salen en unidades de sd y las del ridge en unidades de yield. Para compararlas, las del MLP se multiplican por la sd (celda siguiente). Las correlaciones son comparables directamente.

In [ ]:
y_sd = (lines["BLUE"] - lines.groupby("YEAR")["BLUE"].transform("mean"))[pre].std()
for c in ("gain_all", "oracle_gain_all", "gain_within", "oracle_gain_within"):
    mlp[c] = mlp[c] * y_sd

ridge = pd.read_csv(DATA_DIR / "ridge_temporal_results.csv")
cols = ["test_year", "r_all", "r_within", "gain_all", "gain_within"]
cmp = ridge[cols].merge(mlp[cols], on="test_year", suffixes=("_ridge", "_mlp"))
cmp = cmp[["test_year"] + [f"{m}_{k}" for m in ("r_all", "r_within", "gain_all", "gain_within") for k in ("ridge", "mlp")]]
mlp.to_csv(DATA_DIR / "mlp_temporal_results.csv", index=False)
cmp.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for ax, m in zip(axes, ["r_all", "r_within", "gain_all", "gain_within"]):
    ax.plot(cmp["test_year"], cmp[f"{m}_ridge"], "o-", label="ridge")
    ax.plot(cmp["test_year"], cmp[f"{m}_mlp"], "s-", label="MLP")
    ax.set_title(m)
    ax.axhline(0, color="gray", lw=0.5)
axes[0].legend()
plt.tight_layout()
plt.show()

## 5. Predicciones para 2008 (si 2008 está en `test_years`)

Top 20% predicho, y su rendimiento observado frente a la media de 2008 y frente a lo que daría un top 20% elegido al azar.

In [ ]:
if preds_2008 is not None:
    d = preds_2008.copy()
    k = int(round(len(d) * 0.20))
    top = d.nlargest(k, "pred")
    rand = d.sample(k, random_state=0)
    print(f"Observed BLUE, top 20% predicted: {top['BLUE'].mean():.2f} | random 20%: {rand['BLUE'].mean():.2f} "
          f"| all lines: {d['BLUE'].mean():.2f} | true top 20%: {d.nlargest(k, 'BLUE')['BLUE'].mean():.2f}")
    d.to_csv(DATA_DIR / "mlp_predictions_2008.csv", index=False)
    print(top["POP"].value_counts().head(10))